# Qiskit Fall Fest 2026 — PS-O1 six-node weighted Max-Cut

**Exploratory, reproducible symmetry-reduced A/B comparison (not yet an approved full six-qubit benchmark).** This notebook automatically prepares the repository and executes the experiment from top to bottom.

**Method:** A six-vertex Max-Cut instance is encoded using five logical qubits by fixing vertex 0 (global complement symmetry). This is classically exact but is **not** the full six-qubit QAOA ansatz. Both processor architectures run the **same reduced** QAOA circuit. The required graph size, baseline parameters and results are generated and checked by code, not hard-coded.

Click **Runtime → Run all** in Google Colab. A complete ZIP with the measured report, tables and figures is downloaded at the end. Please do **not submit** until the final run passes and the generated artifacts are pushed to GitHub.

In [ ]:
# Bootstrap: work both in cloned Jupyter repos and in Google Colab's GitHub notebook viewer.
from pathlib import Path
import os, subprocess, sys
ROOT=Path.cwd()
if not (ROOT/'src'/'problem.py').is_file():
    ROOT=Path('/content/qff-2026-ps-o1')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/Abheek-07/qff-2026-ps-o1.git',str(ROOT)],check=True)
    elif (ROOT/'.git').exists():
        subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
    else:
        raise RuntimeError(f'{ROOT} exists but is not a cloned repository')
os.chdir(ROOT)
print('Project root:',ROOT)
print('Installing reproducible dependencies; this can take a few minutes...')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
print('Dependencies installed')

## Independent instance/configuration checks

These tests check the original six-vertex graph's classical optimum, complement symmetry, exact reduction, and the two matching processor JSON generators before the actual quantum benchmark.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True,cwd=ROOT)

## Run the scientific benchmark and produce real exported evidence

This cell optimizes one ideal reduced-QAOA parameter pair, compares Qiskit against an independent NumPy statevector, uses native-basis transpilation at level 1/seed 42, simulates A/B with 4096 shots and provisional synthetic noise, makes routing diagnostics, then generates `results/` and `report.md`.

**Run time:** several minutes, depending on Colab runtime.

In [ ]:
from run_experiment import main
metadata=main()
assert metadata['run_status']=='COMPLETED'
print('FINAL EXPERIMENT SUCCESS — all measurements generated')

## Full six-qubit QAOA on Processor B (capacity-respecting reference)

Run the unmodified six-vertex QAOA with six logical qubits on nine-qubit Processor B using the eight-edge consolidated-kit topology. Processor A has insufficient qubit capacity. This experiment uses provisional noise assumptions and must not be conflated with the preceding **reduced** A/B topology comparison.


In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable, '-m', 'src.full_six_reference'], cwd=ROOT, check=True)
from src.finalize_report import main as finalize_report
finalize_report()
full_data=json.loads((ROOT/'results/full_six_qubit_reference.json').read_text())
assert full_data['status']=='FULL_6Q_EXECUTED'
print('Full six-qubit B benchmark saved and merged into report')


## Export the entire validated project

A ZIP will be created automatically. In Google Colab it will download to your computer. Replace the corresponding GitHub repository files with this ZIP's extracted contents, then commit and push.

Only submit after the GitHub repository includes the generated benchmark CSVs, plots, and the newly written `report.md`.

In [ ]:
from pathlib import Path
import zipfile
ROOT=Path.cwd()
output=ROOT.parent/'qff-2026-ps-o1-validated-results.zip'
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.rglob('*')):
        if not p.is_file() or any(part in {'.git','__pycache__','.ipynb_checkpoints'} for part in p.parts) or p.suffix=='.pyc':
            continue
        z.write(p,arcname=str(Path(ROOT.name)/p.relative_to(ROOT)))
print('Validated export:',output, f'{output.stat().st_size:,} bytes')
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    print('Running outside Colab: ZIP saved above.')